# 05 — Baseline Models
Classical baselines use **aggregated 14-day windows**: for each of the 39 daily features the last
value, mean, max, min, standard deviation and trend (last − first), plus 8 static context features
(242 features). Naive references:

* **Persistence** — tomorrow's class = today's observed class (strong because labels are persistent)
* **Majority class** and **site-month climatology** (most frequent training class for the site and month)

Hyper-parameters (fixed a priori, `scripts/train_baselines.py`): logistic regression (balanced, C=1);
random forest (300 trees, min leaf 5, balanced subsample); HistGradientBoosting (400 iterations,
lr 0.06, 31 leaves, L2 = 1, balanced, early stopping on an internal 10 % split); MLP 256-128
(α = 1e-3, early stopping). All selection uses the **validation** split; test results appear only
in notebook 09. To retrain set `RETRAIN = True`.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
from climatesense.utils import read_json
import platform, torch, sklearn
print("Executed:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "| Python", platform.python_version(),
      "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
def show(p, width=850):
    display(Image(filename=str(ROOT / p), width=width))

Executed: 2026-10-25 18:06 | Python 3.13.7 | torch 2.14.1+cu126 | sklearn 1.9.1 | pandas 3.1.0


In [2]:
RETRAIN = False
if RETRAIN:
    import subprocess; subprocess.run([sys.executable, str(ROOT / "scripts/train_baselines.py")], check=True)
rows = []
for f in sorted((ROOT / "experiments/baseline").glob("baseline-*_h1.json")):
    r = read_json(f); m = r["val_metrics"]
    rows.append(dict(model=r["model"], val_accuracy=m["accuracy"], val_balanced_accuracy=m["balanced_accuracy"], val_macro_f1=m["macro_f1"],
                     val_weighted_f1=m["weighted_f1"], val_roc_auc=m["roc_auc_macro_ovr"], train_time_s=r["train_time_s"], n_params=r["n_params"]))
s = read_json(ROOT / "experiments/baseline/summary_L14_h1.json")
for k in ["persistence_L14_h1", "majority_L14_h1", "climatology_L14_h1"]:
    rows.append(dict(model=k.split("_")[0], val_accuracy=s[k]["val_accuracy"], val_balanced_accuracy=s[k]["val_balanced_accuracy"], val_macro_f1=s[k]["val_macro_f1"]))
pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False).round(4)

,model,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,val_roc_auc,train_time_s,n_params
3,random_forest,0.8427,0.7448,0.7405,0.8432,0.9549,63.0108,4016074.0
0,hist_gradient_boosting,0.8380,0.7524,0.7397,0.8399,0.9538,116.9899,30500.0
1,logreg,0.8349,0.7354,0.7291,0.8359,0.9480,33.5362,948.0
2,mlp,0.8361,0.7135,0.7181,0.8311,0.9522,54.2020,94084.0
4,persistence,0.8242,0.7153,0.7160,NaN,NaN,NaN,NaN
6,climatology,0.6226,0.5957,0.5276,NaN,NaN,NaN,NaN
5,majority,0.6520,0.2500,0.1973,NaN,NaN,NaN,NaN


### Validation classification report of the strongest classical baseline

In [3]:
from climatesense.labeling.compound_risk import CLASS_NAMES
d = np.load(ROOT / "results/predictions/baseline-hist_gradient_boosting_L14_h1_val.npz")
from sklearn.metrics import classification_report
print(classification_report(d["y"], d["proba"].argmax(1), target_names=CLASS_NAMES, digits=3))

              precision    recall  f1-score   support

         Low      0.938     0.918     0.928     14007
    Moderate      0.675     0.679     0.677      4371
        High      0.529     0.583     0.555      1598
    Critical      0.771     0.830     0.799      1508

    accuracy                          0.838     21484
   macro avg      0.728     0.752     0.740     21484
weighted avg      0.842     0.838     0.840     21484



**Reading.** Gradient boosting and the random forest clearly beat the naive baselines, but the
margin over persistence is modest: most of the predictable signal is that compound-risk states
persist from one day to the next.